# 실습 02 · 회귀로 화합물 용해도 예측하기
### "이 분자가 물에 얼마나 녹을까?" 를 AI 로 예측

**왜 중요한가 (현장 맥락)**
신약 후보물질이 약이 되려면 **물에 적당히 녹아야** 흡수됩니다.
실제로 합성·측정하려면 시간과 돈이 많이 드므로, **구조만 보고 용해도를 미리 예측**하면
합성 우선순위를 정하는 데 큰 도움이 됩니다. 이것이 제약사의 대표적 AI 활용 사례입니다.

**이 노트북의 흐름**
1. SMILES(분자 글자표현) → **분자 지문/특성값(descriptor)** 으로 변환 (RDKit)
2. 특성값 → 용해도를 예측하는 **회귀 모델** 학습
3. 예측이 얼마나 정확한지 **평가(R², RMSE)** 하고 그림으로 확인


## 1. RDKit 설치
**RDKit** 은 신약개발 현장에서 표준으로 쓰는 화학정보학 라이브러리입니다.
분자를 다루고, 구조를 그리고, 수백 가지 물리화학적 특성을 계산해 줍니다.
Colab 에서는 아래 한 줄로 설치합니다. (약 30초~1분)


In [ ]:
!pip install rdkit -q
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, Crippen
print("RDKit 준비 완료 ✅")

In [ ]:
# 데이터 불러오기 (예제 01 과 동일한 용해도 데이터)
import pandas as pd, numpy as np
url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/delaney-processed.csv"
df = pd.read_csv(url)
df = df.rename(columns={"measured log solubility in mols per litre":"logS"})
print("분자 개수:", len(df))
df[["Compound ID","smiles","logS"]].head()

## 2. 분자를 눈으로 보기 (SMILES → 구조 그림)
SMILES 는 분자를 글자로 적은 것입니다. RDKit 이 이를 **실제 분자 그림**으로 그려줍니다.


In [ ]:
# 데이터의 앞 8개 분자를 그림으로
mols = [Chem.MolFromSmiles(s) for s in df["smiles"].head(8)]
Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(200,160),
                     legends=list(df["Compound ID"].head(8)))

## 3. ⭐ 특성 추출(Feature Engineering)
머신러닝 모델은 글자(SMILES)를 직접 이해하지 못합니다.
분자를 **숫자 특성들**로 바꿔줘야 합니다. 용해도와 관련 깊은 대표 특성 몇 개를 계산합니다.

| 특성 | 의미 | 용해도와의 관계 |
|---|---|---|
| MolWt | 분자량 | 클수록 잘 안 녹는 경향 |
| LogP | 지용성(기름 좋아하는 정도) | 높을수록 물에 안 녹음 |
| TPSA | 극성 표면적 | 클수록 물에 잘 녹음 |
| NumHDonors/Acceptors | 수소결합 주개/받개 수 | 물과 결합, 용해도↑ |
| NumRotatableBonds | 회전 가능한 결합 수 | 유연성 |
| AromaticRings | 방향족 고리 수 | 많으면 잘 안 녹음 |


In [ ]:
# 각 분자에 대해 6가지 특성을 계산하는 함수
def featurize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:        # 이상한 SMILES 는 건너뜀
        return None
    return {
        "MolWt"      : Descriptors.MolWt(mol),          # 분자량
        "LogP"       : Crippen.MolLogP(mol),            # 지용성
        "TPSA"       : Descriptors.TPSA(mol),           # 극성 표면적
        "HDonors"    : Descriptors.NumHDonors(mol),     # 수소결합 주개
        "HAcceptors" : Descriptors.NumHAcceptors(mol),  # 수소결합 받개
        "RotBonds"   : Descriptors.NumRotatableBonds(mol),
        "AromRings"  : Descriptors.NumAromaticRings(mol),
    }

# 모든 분자에 적용 → 특성 표(X) 만들기
feat = df["smiles"].apply(featurize).apply(pd.Series)
data = pd.concat([feat, df["logS"]], axis=1).dropna()
X = data.drop(columns="logS")   # 입력(특성)
y = data["logS"]                # 정답(용해도)
print("특성 표 크기:", X.shape)
X.head()

In [ ]:
# 어떤 특성이 용해도와 관련이 깊은지 상관관계로 확인 (LogP 가 강한 음의 상관!)
corr = data.corr()["logS"].sort_values()
print("각 특성과 용해도(logS)의 상관계수:")
print(corr)

## 4. 학습/평가 데이터 나누기
모델을 **처음 보는 데이터**에서도 잘 맞추는지 확인해야 합니다.
그래서 데이터를 **학습용(80%)** 과 **평가용(20%)** 으로 나눕니다.
> 이 원칙(학습/평가 분리)은 모든 실무 AI 프로젝트의 기본입니다.


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)   # random_state: 매번 같은 결과 나오게 고정
print("학습:", X_train.shape[0], "개 / 평가:", X_test.shape[0], "개")

## 5. 회귀 모델 3종 비교
- **선형회귀**: 가장 단순, 해석 쉬움 (특성들의 가중합)
- **랜덤포레스트**: 여러 결정트리를 합친 강력한 모델, 비선형 관계도 학습
- **그래디언트부스팅**: 대회에서 자주 우승하는 강한 모델

세 모델을 같은 데이터로 학습해 **누가 더 정확한지** 비교합니다.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

models = {
    "선형회귀"       : LinearRegression(),
    "랜덤포레스트"   : RandomForestRegressor(n_estimators=300, random_state=42),
    "그래디언트부스팅": GradientBoostingRegressor(random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)             # 학습
    pred = model.predict(X_test)            # 평가데이터 예측
    r2   = r2_score(y_test, pred)           # 1에 가까울수록 좋음
    rmse = mean_squared_error(y_test, pred) ** 0.5  # 작을수록 좋음
    results[name] = (r2, rmse, pred)
    print(f"{name:14s}  R²={r2:.3f}   RMSE={rmse:.3f}")

**해석**
- **R²(결정계수)**: 1에 가까울수록 좋음. 0.8 이면 용해도 변동의 80%를 설명.
- **RMSE**: 예측 오차의 크기(작을수록 좋음), 용해도 단위와 같음.

보통 랜덤포레스트/부스팅이 선형회귀보다 좋습니다 (비선형 관계를 잡아내므로).


In [ ]:
# 가장 좋은 모델의 '예측 vs 실제' 산점도 (점들이 대각선에 가까울수록 정확)
import matplotlib.pyplot as plt
best = max(results, key=lambda k: results[k][0])
r2, rmse, pred = results[best]

plt.figure(figsize=(5.5,5.5))
plt.scatter(y_test, pred, alpha=0.5, color="#4C72B0")
lim = [y.min()-0.5, y.max()+0.5]
plt.plot(lim, lim, "r--", label="완벽한 예측")
plt.xlabel("실제 용해도"); plt.ylabel("예측 용해도")
plt.title(f"{best}  (R²={r2:.3f})"); plt.legend(); plt.show()

## 6. 어떤 특성이 예측에 중요했나 (설명가능성)
현장에서는 "왜 이렇게 예측했는가?"가 중요합니다.
랜덤포레스트는 각 특성의 **중요도**를 알려줍니다.


In [ ]:
rf = models["랜덤포레스트"]
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
imp.plot.barh(figsize=(6,4), color="#55A868")
plt.title("특성 중요도 (용해도 예측)"); plt.xlabel("중요도"); plt.show()
print("→ LogP(지용성)와 분자량이 용해도 예측의 핵심 요인임을 확인")

## 7. 새로운 분자로 예측해보기 (실전 활용)
학습된 모델로, **아직 측정 안 한 분자**의 용해도를 예측해봅니다.
카페인과 이부프로펜을 넣어보겠습니다.


In [ ]:
test_molecules = {
    "카페인"    : "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "이부프로펜": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "아스피린"  : "CC(=O)Oc1ccccc1C(=O)O",
}
rows = []
for name, smi in test_molecules.items():
    f = featurize(smi)
    pred = models["랜덤포레스트"].predict(pd.DataFrame([f])[X.columns])[0]
    rows.append({"분자": name, "예측 logS": round(pred,2)})
pd.DataFrame(rows)

## 정리 & 현장 응용
- SMILES → RDKit 특성 → 회귀모델 → 용해도 예측, **실제 제약 파이프라인의 축소판**
- 같은 틀로 **LogP, 독성, 흡수율, 대사안정성** 등 어떤 물성이든 예측 가능
- 데이터의 `logS` 열만 다른 물성으로 바꾸면 그대로 재사용됩니다.

**면접에서 말할 수 있는 한 문장**:
"RDKit 로 분자 descriptor 를 뽑고 랜덤포레스트 회귀로 용해도를 예측해봤습니다.
LogP 가 가장 중요한 특성이었고 R²는 약 0.8 이었습니다."
